# After-Sales Customer Support Portal
### Automated Invoice Extraction & Feedback Sentiment Analysis

**Student:** Tanmay Gaikwad | **Roll No:** A4-63

---

A customer visits the support portal, uploads their purchase invoice (PDF or text), and submits their review or complaint. The system automatically:
- Extracts invoice details (company, date, amount, items)
- Analyzes and classifies the feedback sentiment
- Provides a complete NLP analysis of the input

In [ ]:
# === Setup & Model Training ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet as wn
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from collections import Counter

import spacy
nlp = spacy.load("en_core_web_sm")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from wordcloud import WordCloud

!pip install pdfplumber -q
import pdfplumber

# Load dataset and train models
df = pd.read_csv('dataset/customer_invoice_feedback.csv')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
lm = WordNetLemmatizer()

X = df['customer_feedback'].astype(str)
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

nb_model = make_pipeline(TfidfVectorizer(stop_words='english'), MultinomialNB())
nb_model.fit(X_train, y_train)

lr_model = make_pipeline(TfidfVectorizer(stop_words='english'), LogisticRegression(max_iter=1000, random_state=42))
lr_model.fit(X_train, y_train)

nb_acc = accuracy_score(y_test, nb_model.predict(X_test))
lr_acc = accuracy_score(y_test, lr_model.predict(X_test))

print("System ready!")
print(f"Models trained on {len(X_train)} samples | NB Accuracy: {nb_acc:.2%} | LR Accuracy: {lr_acc:.2%}")

## Enter Your Invoice & Feedback

In [ ]:
print("=" * 60)
print("   AFTER-SALES CUSTOMER SUPPORT PORTAL")
print("=" * 60)

print("\nHow would you like to provide the invoice?")
print("  1 - Upload a PDF file (provide file path)")
print("  2 - Paste invoice text manually")
choice = input("Enter choice (1 or 2): ").strip()

user_invoice = ""

if choice == "1":
    pdf_path = input("\nEnter the full path to your invoice PDF: ").strip()
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages_text = []
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    pages_text.append(text)
            user_invoice = '\n'.join(pages_text)
        print(f"\nExtracted text from {len(pdf.pages)} page(s):")
        print(user_invoice[:500])
        if len(user_invoice) > 500:
            print("... [truncated]")
    except FileNotFoundError:
        print(f"File not found: {pdf_path}. Using sample invoice.")
    except Exception as e:
        print(f"Error: {e}. Using sample invoice.")
elif choice == "2":
    print("\nPaste your invoice text (press Enter twice when done):")
    lines = []
    while True:
        line = input()
        if line == "":
            break
        lines.append(line)
    user_invoice = '\n'.join(lines)

if not user_invoice.strip():
    user_invoice = "Invoice No: INV-999 | Date: 2026-04-10 | From: TechNova Solutions Pvt Ltd | To: Sharma Electronics, Mumbai | Items: 2x Mechanical Keyboard @Rs.2500, 1x Gaming Mouse @Rs.1800 | Subtotal: Rs.6800 | GST (18%): Rs.1224 | Total: Rs.8024 | Payment Due: 2026-05-10"
    print(f"\nUsing sample invoice:\n{user_invoice}")

print("\n" + "-" * 60)
user_feedback = input("\nEnter your review/complaint: ").strip()

if not user_feedback:
    user_feedback = "The mechanical keyboard is absolutely fantastic! Great build quality and very responsive keys. However, the mouse scroll wheel feels a bit loose. Overall a good purchase from TechNova."
    print(f"Using sample: {user_feedback}")

print("\nInput received! Run the next cell to see results.")

## Analysis Results

In [ ]:
print("=" * 60)
print("   1. INVOICE DATA EXTRACTION")
print("=" * 60)

# Regex extraction
invoice_no = re.findall(r'INV-\d+', user_invoice)
dates = re.findall(r'\d{4}-\d{2}-\d{2}', user_invoice)
amounts = re.findall(r'Rs\.?[\d,]+(?:\.\d{2})?', user_invoice)
from_company = re.findall(r'From:\s*([^|\n]+)', user_invoice)
to_company = re.findall(r'To:\s*([^|\n]+)', user_invoice)
items = re.findall(r'Items:\s*([^|\n]+)', user_invoice)

print("\n[Regex Extraction]")
print(f"  Invoice No: {invoice_no[0] if invoice_no else 'N/A'}")
print(f"  From:       {from_company[0].strip() if from_company else 'N/A'}")
print(f"  To:         {to_company[0].strip() if to_company else 'N/A'}")
print(f"  Dates:      {dates if dates else 'N/A'}")
print(f"  Items:      {items[0].strip() if items else 'N/A'}")
print(f"  Amounts:    {amounts if amounts else 'N/A'}")

# spaCy NER
print("\n[Named Entity Recognition - Invoice]")
doc_inv = nlp(user_invoice)
if doc_inv.ents:
    print(f"  {'Entity':<35} {'Type':<15}")
    print(f"  {'-'*48}")
    for ent in doc_inv.ents:
        print(f"  {ent.text:<35} {ent.label_:<15}")
else:
    print("  No named entities detected")

print("\n[Named Entity Recognition - Feedback]")
doc_fb = nlp(user_feedback)
if doc_fb.ents:
    print(f"  {'Entity':<35} {'Type':<15}")
    print(f"  {'-'*48}")
    for ent in doc_fb.ents:
        print(f"  {ent.text:<35} {ent.label_:<15}")
else:
    print("  No named entities detected")

In [ ]:
print("=" * 60)
print("   2. TEXT PREPROCESSING")
print("=" * 60)

tokens = word_tokenize(user_feedback)
sentences = sent_tokenize(user_feedback)
cleaned = [w for w in tokens if w.lower().isalpha() and w.lower() not in stop_words]

print(f"\n  Sentences:       {len(sentences)}")
for i, s in enumerate(sentences, 1):
    print(f"    {i}. {s}")

print(f"\n  Total Tokens:    {len(tokens)}")
print(f"  After Cleaning:  {len(cleaned)}")
print(f"  Cleaned Words:   {cleaned}")

print(f"\n  {'Word':<18} {'Stemmed':<18} {'Lemmatized':<18}")
print(f"  {'-'*52}")
for w in cleaned[:12]:
    print(f"  {w:<18} {ps.stem(w):<18} {lm.lemmatize(w.lower()):<18}")

print(f"\n  POS Tags:")
print(f"  {'Word':<18} {'Tag':<8}")
print(f"  {'-'*24}")
for word, tag in pos_tag(tokens):
    print(f"  {word:<18} {tag:<8}")

In [ ]:
print("=" * 60)
print("   3. FEEDBACK ANALYSIS")
print("=" * 60)

# Word frequency
word_freq_input = Counter(cleaned)
print("\n  Word Frequencies:")
for word, count in word_freq_input.most_common(15):
    bar = '*' * (count * 5)
    print(f"    {word:<18} {count:>3} {bar}")

# N-grams
fb_tokens = [w.lower() for w in word_tokenize(re.sub(r'[^\w\s]', '', user_feedback.lower()))]
bigrams_input = [(fb_tokens[i], fb_tokens[i+1]) for i in range(len(fb_tokens)-1)]
trigrams_input = [(fb_tokens[i], fb_tokens[i+1], fb_tokens[i+2]) for i in range(len(fb_tokens)-2)]

print(f"\n  Top Bigrams:  {[' '.join(g) for g, _ in Counter(bigrams_input).most_common(5)]}")
print(f"  Top Trigrams: {[' '.join(g) for g, _ in Counter(trigrams_input).most_common(5)]}")

# WordCloud
if cleaned:
    fig, ax = plt.subplots(figsize=(10, 4))
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap='viridis', min_font_size=10).generate(' '.join(cleaned))
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title('Feedback Word Cloud', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
print("=" * 60)
print("   4. SENTIMENT PREDICTION")
print("=" * 60)

nb_pred_user = nb_model.predict([user_feedback])[0]
lr_pred_user = lr_model.predict([user_feedback])[0]
nb_proba_user = nb_model.predict_proba([user_feedback])[0]
lr_proba_user = lr_model.predict_proba([user_feedback])[0]

print(f"\n  Naive Bayes:          {nb_pred_user.upper()}")
print(f"    Confidence: {dict(zip(nb_model.classes_, [f'{p:.2%}' for p in nb_proba_user]))}")
print(f"\n  Logistic Regression:  {lr_pred_user.upper()}")
print(f"    Confidence: {dict(zip(lr_model.classes_, [f'{p:.2%}' for p in lr_proba_user]))}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_map = {'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#3498db'}

for idx, (name, proba, classes) in enumerate([
    ('Naive Bayes', nb_proba_user, nb_model.classes_),
    ('Logistic Regression', lr_proba_user, lr_model.classes_)
]):
    bar_colors = [colors_map.get(c, 'gray') for c in classes]
    axes[idx].bar(classes, proba, color=bar_colors)
    axes[idx].set_title(f'{name}')
    axes[idx].set_ylabel('Probability')
    axes[idx].set_ylim(0, 1)
    for i, v in enumerate(proba):
        axes[idx].text(i, v + 0.02, f'{v:.2%}', ha='center', fontweight='bold')

plt.suptitle('Sentiment Prediction', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("   ANALYSIS COMPLETE")
print("=" * 60)
print(f"""
  INVOICE:
    Invoice No:  {invoice_no[0] if invoice_no else 'N/A'}
    Company:     {from_company[0].strip() if from_company else 'N/A'}
    Customer:    {to_company[0].strip() if to_company else 'N/A'}
    Amount:      {amounts[-1] if amounts else 'N/A'}

  FEEDBACK:
    Words: {len(tokens)} | Cleaned: {len(cleaned)} | Sentences: {len(sentences)}
    Top Words:  {', '.join([w for w, _ in word_freq_input.most_common(5)])}
    Top Bigram: {' '.join(Counter(bigrams_input).most_common(1)[0][0]) if bigrams_input else 'N/A'}

  SENTIMENT:
    Naive Bayes:          {nb_pred_user.upper()}
    Logistic Regression:  {lr_pred_user.upper()}

  PRIORITY: {'HIGH - Route to support team immediately' if lr_pred_user == 'negative' else 'NORMAL - Standard processing' if lr_pred_user == 'neutral' else 'LOW - Positive feedback logged'}
""")

---
### Conclusion

This **After-Sales Customer Support Portal** uses NLP to automate:
1. **Invoice extraction** -- Regex + spaCy NER to pull structured data from invoices
2. **Text preprocessing** -- Tokenization, stemming, lemmatization for clean analysis
3. **Feedback analysis** -- Word frequency, n-grams, and word clouds to understand patterns
4. **Sentiment classification** -- TF-IDF + Naive Bayes/Logistic Regression to prioritize complaints

**Libraries:** NLTK, spaCy, scikit-learn, pandas, matplotlib, WordCloud, pdfplumber